In [6]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

data = np.load("20000.npy")
df = pd.DataFrame(data)
df["palya_id"] = df["eventID"].astype(str) + "_" + df["trackID"].astype(str)

parameters = ['posX', 'posY', 'edep'] 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128

class Matcher(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim) 
        )
    def forward(self, x):
        return self.net(x)

def calculate_accuracy_mm(pred, target, mins, maxs, threshold_mm=2.0):
    # visszaalakítás eredeti skálára (mm-be)
    scale_factor = (maxs - mins)
    pred_mm = pred * scale_factor
    target_mm = target * scale_factor
    
    dist_mm = torch.norm(pred_mm - target_mm, dim=1)
    return (dist_mm < threshold_mm).float().mean().item()

def prepare_data(layerA, layerB, params):
    common_labels = np.array(list(set(layerA["palya_id"]) & set(layerB["palya_id"])))
    np.random.shuffle(common_labels)
    train_split = int(0.9 * len(common_labels))
    
    # skálázási paraméterek
    dfA_subset = layerA.set_index("palya_id").loc[common_labels[:train_split]]
    mins, maxs = torch.tensor(dfA_subset[params].min().values).to(device), torch.tensor(dfA_subset[params].max().values).to(device)
    
    def scale(df_subset):
        return (torch.tensor(df_subset[params].values).to(device) - mins) / (maxs - mins + 1e-6)

    l_train = [scale(layerA.set_index("palya_id").loc[common_labels[:train_split]]), 
               scale(layerB.set_index("palya_id").loc[common_labels[:train_split]])]
    l_val = [scale(layerA.set_index("palya_id").loc[common_labels[train_split:]]), 
             scale(layerB.set_index("palya_id").loc[common_labels[train_split:]])]
    return l_train, l_val, mins, maxs

# tanítás
def train(l_train, l_val, mins, maxs, name):
    print(f"\n Tanítás: {name} ")
    model = Matcher(len(parameters)).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.MSELoss()
    
    for e in range(300):
        model.train()
        indices = torch.randperm(len(l_train[0]))
        train_accs = []
        for i in range(0, len(indices), batch_size):
            batch_idx = indices[i:i+batch_size]
            x_prev, x_next = l_train[0][batch_idx], l_train[1][batch_idx]
            
            optimizer.zero_grad()
            pred = model(x_prev)
            loss = criterion(pred, x_next)
            loss.backward()
            optimizer.step()
            train_accs.append(calculate_accuracy_mm(pred, x_next, mins, maxs))
        
        model.eval()
        with torch.no_grad():
            v_acc = calculate_accuracy_mm(model(l_val[0]), l_val[1], mins, maxs)
        
        if e % 20 == 0:
            print(f"Epoch {e:3d} | Train Acc: {np.mean(train_accs)*100:6.2f}% | Val Acc: {v_acc*100:6.2f}%")
    return model

pairs = [([1, 2], "L1-L2"), ([2, 3], "L2-L3")]
models = {}

for indices, name in pairs:
    lA = df[df["volumeID[3]"] == indices[0]].drop_duplicates(subset=["palya_id"])
    lB = df[df["volumeID[3]"] == indices[1]].drop_duplicates(subset=["palya_id"])
    
    train_data, val_data, mins, maxs = prepare_data(lA, lB, parameters)
    models[name] = train(train_data, val_data, mins, maxs, name)


 Tanítás: L1-L2 
Epoch   0 | Train Acc:   2.19% | Val Acc:   4.33%
Epoch  20 | Train Acc:  90.41% | Val Acc:  90.57%
Epoch  40 | Train Acc:  89.80% | Val Acc:  91.17%
Epoch  60 | Train Acc:  89.80% | Val Acc:  90.81%
Epoch  80 | Train Acc:  90.10% | Val Acc:  90.52%
Epoch 100 | Train Acc:  90.02% | Val Acc:  90.75%
Epoch 120 | Train Acc:  89.78% | Val Acc:  87.67%
Epoch 140 | Train Acc:  89.85% | Val Acc:  91.35%
Epoch 160 | Train Acc:  89.78% | Val Acc:  90.87%
Epoch 180 | Train Acc:  89.88% | Val Acc:  88.62%
Epoch 200 | Train Acc:  90.58% | Val Acc:  90.04%
Epoch 220 | Train Acc:  89.64% | Val Acc:  91.46%
Epoch 240 | Train Acc:  90.13% | Val Acc:  91.46%
Epoch 260 | Train Acc:  90.03% | Val Acc:  90.40%
Epoch 280 | Train Acc:  89.18% | Val Acc:  91.58%

 Tanítás: L2-L3 
Epoch   0 | Train Acc:   3.20% | Val Acc:   7.71%
Epoch  20 | Train Acc:  97.19% | Val Acc:  97.49%
Epoch  40 | Train Acc:  97.03% | Val Acc:  97.19%
Epoch  60 | Train Acc:  96.76% | Val Acc:  96.89%
Epoch  80 | Tr